In [96]:
import pandas as pd
import geopandas as gpd
import altair as alt
from shapely.geometry import Point

df = pd.read_csv('../../data/Taxi_Trips.csv')
geometry = [Point(xy) for xy in zip(df['Pickup Centroid Longitude'], df['Pickup Centroid Latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=4326)

In [136]:
df['Trip Start Timestamp'] = pd.to_datetime(df['Trip Start Timestamp'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df = df.dropna(subset=['Trip Start Timestamp'])  # Remove rows where parsing failed

In [137]:
df['DayOfWeek'] = df['Trip Start Timestamp'].dt.day_name()

In [138]:
agg_df = df.groupby('DayOfWeek')['Fare'].agg([
    ('min', 'min'),
    ('q1', lambda x: x.quantile(0.25)),
    ('median', 'median'),
    ('q3', lambda x: x.quantile(0.75)),
    ('max', 'max')
]).reset_index()

In [139]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
agg_df['DayOfWeek'] = pd.Categorical(agg_df['DayOfWeek'], categories=weekday_order, ordered=True)

# Now sort based on this order
agg_df = agg_df.sort_values(by=['DayOfWeek'])

In [140]:
agg_df

,DayOfWeek,min,q1,median,q3,max
1,Monday,0.00,9.18,15.75,34.5000,405.50
5,Tuesday,0.00,7.50,13.25,31.7500,5000.06
6,Wednesday,0.00,7.50,12.00,30.0000,4500.36
4,Thursday,0.00,7.50,12.50,31.0000,9007.65
0,Friday,0.00,7.75,12.50,30.0000,6001.35
2,Saturday,3.25,10.00,15.25,29.4375,130.00
3,Sunday,0.00,8.25,12.99,29.5000,620.01


In [141]:
spec = """
{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "data": {"values": %s},
  "encoding": {"y": {"field": "DayOfWeek", "type": "nominal", "sort": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]}},
  "layer": [
    {
      "mark": {"type": "bar", "size": 14},
      "encoding": {
        "x": {"field": "q1", "type": "quantitative"},
        "x2": {"field": "q3"}
      }
    },
    {
      "mark": {"type": "tick", "color": "black", "size": 18},
      "encoding": {
        "x": {"field": "median", "type": "quantitative"}
      }
    }
  ]
}

"""%agg_df.to_json(orient='records')

In [142]:
chart = alt.Chart.from_json(spec)
chart

alt.LayerChart(...)

In [143]:
df['Hour'] = df['Trip Start Timestamp'].dt.hour

agg_df = df.groupby('Hour')['Fare'].agg([
    ('min', 'min'),
    ('q1', lambda x: x.quantile(0.25)),
    ('median', 'median'),
    ('q3', lambda x: x.quantile(0.75)),
    ('max', 'max')
]).reset_index()

In [144]:
agg_df

,Hour,min,q1,median,q3,max
0,0,0.00,9.500,16.000,33.5000,246.00
1,1,0.00,9.450,15.000,30.2500,173.50
2,2,3.25,8.750,13.000,26.7500,144.50
3,3,0.05,9.450,18.000,34.2500,325.00
4,4,0.00,11.625,29.250,43.3750,175.00
5,5,3.25,12.110,31.000,43.5000,160.50
6,6,2.22,9.450,26.000,39.2500,180.50
7,7,0.01,7.750,14.505,32.7500,440.00
8,8,0.00,7.250,11.520,29.5000,149.75
9,9,0.00,7.250,12.000,30.2500,180.00


In [145]:
spec = """
{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "data": {"values": %s},
  "layer": [
    {
      "mark": "errorband",
      "encoding": {
        "y": {
          "field": "q3",
          "type": "quantitative",
          "scale": {"zero": false}
        },
        "y2": {"field": "q1"},
        "x": {
          "field": "Hour"
        }
      }
    },
    {
      "mark": "line",
      "encoding": {
        "y": {
          "field": "median",
          "type": "quantitative"
        },
        "x": {
          "field": "Hour"
        }
      }
    }
  ]
}

"""%agg_df.to_json(orient='records')

In [146]:
chart = alt.Chart.from_json(spec)
chart

alt.LayerChart(...)